In [5]:
import pandas as pd
import plotly.express as px


In [6]:
nuevos_nombres = [
    'estado', 'codigo', 'fecha_pedido', 'fecha_eliminacion', 
    'mesa', 'mozo', 'observaciones', 'unidad', 'producto', 
    'cantidad', 'precio_unitario', 'total'
]
numericos=['cantidad', 'precio_unitario', 'total']

df = pd.read_excel(
    "Pedidos_test.xlsx", 
    sheet_name="Reporte Pedidos detallado", 
    usecols="A:L",
    engine="calamine",
    header=0,              
    names=nuevos_nombres   
)
df = (df.assign(
    fecha_pedido = lambda x: pd.to_datetime(x['fecha_pedido'], format='%d/%m/%Y %H:%M:%S', errors='coerce'),
    fecha_eliminacion = lambda x: pd.to_datetime(x['fecha_eliminacion'], format='%d/%m/%Y %H:%M:%S', errors='coerce'),
    f_pedido = lambda x: x['fecha_pedido'].dt.strftime('%d/%m/%Y'), 
    h_pedido = lambda x: x['fecha_pedido'].dt.strftime('%H:%M:%S'),
    f_eliminacion = lambda x: x['fecha_eliminacion'].dt.strftime('%d/%m/%Y'), 
    h_eliminacion = lambda x: x['fecha_eliminacion'].dt.strftime('%H:%M:%S')
).dropna( # filtro nulas (ultima fila q totaliza)
    subset=['estado']
).assign(**{col: df[col].apply(pd.to_numeric, errors='coerce') for col in numericos})
)



In [9]:
tickets = df[df['estado'] == 'Facturado'].groupby(['mesa', 'codigo'])['total'].sum().reset_index()
# Luego sacamos el promedio de esos tickets por mesa
ticket_promedio_mesa = tickets.groupby('mesa')['total'].mean().round(2).reset_index()
ticket_promedio_mesa.rename(columns={'total': 'Ticket Promedio'}, inplace=True)
ticket_promedio_mesa

,mesa,Ticket Promedio
0,P1_M10,50.0
1,P1_MESA1,29.5
2,P1_MESA2,25.0
3,P1_MESA3,40.0
4,P1_MESA4,17.5
5,P2_MESA2,21.0
6,SIN MESA,20.5


In [18]:
ratios_estado = df['estado'].value_counts().reset_index()
ratios_estado.columns = ['Estado', 'Total']
ratios_estado

,Estado,Total
0,Facturado,25
1,Pendiente,17


In [11]:
codigos_delivery = df[df['producto'].str.contains('DELIVERY', case=False, na=False)]['codigo'].unique()
len(codigos_delivery)

1

In [12]:
tabla_contraste = (df[df['estado'] == 'Facturado']
    .groupby(['h_pedido', 'mesa', 'codigo'])
    .agg(
        Productos=('producto', lambda x: ', /n'.join(x.dropna())),
        Total=('total', 'sum')
    )
    .reset_index()
    .sort_values(by=['h_pedido', 'mesa'])
)
tabla_contraste

,h_pedido,mesa,codigo,Productos,Total
0,09:34:36,SIN MESA,P2605-317,"CAFE + POLLO, /nDELIVERY",14
1,17:28:59,P1_MESA4,P2605-318,CAFE + CHORIPAN,9
2,17:36:44,P1_MESA1,P2605-319,"JUGO PAPAYA, /nPAN CON POLLO, /nCAFE + POLLO",18
3,18:05:34,P1_MESA2,P2605-320,"CAFE + POLLO, /nALITAS BBQ",25
4,18:35:46,P2_MESA2,P2605-321,"SALCHIPAPA SIMPLE, /nCAFE PASADO, /nCAFE PASADO",21
5,19:02:28,SIN MESA,P2605-322,"CESAR DE CASA, /nFRAPUCCINO CLASICO",27
6,19:47:53,P1_MESA4,P2605-324,"ARANDANO CON LECHE, /nALITAS BBQ",26
7,19:49:36,P1_MESA1,P2605-325,"LLUVIA DE ALITAS, /nTAPER PARA LLEVAR",41
8,19:52:55,P1_M10,P2605-326,"MARACUYA REFRESC, /nLLUVIA DE ALITAS",50
9,20:18:37,P1_MESA3,P2605-328,"JUGO NARANJA, /nJUGO PAPAYA, /nJUGO PAPAYA, /n...",40


In [13]:
tabla_eliminacion = df[df['fecha_eliminacion'].notna() | (df['estado'] == 'Anulado')].copy()
tabla_eliminacion = tabla_eliminacion[['h_pedido', 'h_eliminacion', 'mesa', 'producto', 'total']]
tabla_eliminacion

,h_pedido,h_eliminacion,mesa,producto,total


In [22]:
orden_horas = [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 0, 1, 2, 3]


orden_mesas = [
    'P1_MESA1', 'P1_MESA2', 'P1_MESA3', 'P1_MESA4', 'P1_MESA5', 'P1_MESA6', 'P1_MESA7', 'P1_MESA8',
    'P1_MESA9', 'P1_M10', 'P1_M11', 'P1_M12',
    'P2_MESA1','P2_MESA2','P2_MESA3','P2_MESA4','P2_MESA5',
    'SIN MESA'
]

In [47]:
pivot_pedidos = df[df['estado'] == 'Facturado'].pivot_table(
    index='hora_entera', 
    columns='mesa', 
    values='codigo', 
    aggfunc='nunique', 
    fill_value=''
)

pivot_pedidos = pivot_pedidos.reindex(index=orden_horas, columns=orden_mesas)

etiquetas_horas = [f"{h}:00" for h in pivot_pedidos.index]


fig_pedidos = px.imshow(
    pivot_pedidos,
    labels=dict(x="Mesa", y="Hora", color="Total pedidos"),
    x=pivot_pedidos.columns,
    y=etiquetas_horas,
    text_auto="1f",    
    aspect="auto",
    color_continuous_scale="reds",
    title="Total de Pedidos (Hora vs Mesa)",
    width=700,         # Ancho amplio para que las mesas entren rectas
    height=600          # Alto suficiente para que no se oculte ninguna hora
)
fig_pedidos.update_xaxes(
    side='bottom',                 # Pone las mesas en la parte de arriba
    tickangle=270,                # Fuerza a que las etiquetas estén completamente horizontales
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las mesas de la lista
    tickvals=list(range(len(orden_mesas))),
    ticktext=orden_mesas
)

# 6. Configuración del Eje Y (Horas) - COMPLETO Y SIN LÍNEAS
fig_pedidos.update_yaxes(
    type='category',
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las horas de la lista
    tickvals=list(range(len(etiquetas_horas))),
    ticktext=etiquetas_horas
)

# 7. Añadir líneas separadoras de jornada
fig_pedidos.add_hline(y=3.5, line_dash="dash", line_color="gray", annotation_text=" Inicio PM (Mediodía)", annotation_position="bottom right")
fig_pedidos.add_hline(y=15.5, line_dash="dash", line_color="gray", annotation_text=" Madrugada", annotation_position="bottom right")

fig_pedidos.show()

In [ ]:
pivot_dinero = df[df['estado'] == 'Facturado'].pivot_table(
    index='hora_entera', 
    columns='mesa', 
    values='total', 
    aggfunc='sum'
)

pivot_dinero = pivot_dinero.reindex(index=orden_horas, columns=orden_mesas)
etiquetas_horas = [f"{h}:00" for h in pivot_dinero.index]


fig_dinero = px.imshow(
    pivot_dinero,
    labels=dict(x="Mesa", y="Hora", color="Total(S/.)"),
    x=pivot_dinero.columns,
    y=etiquetas_horas,
    text_auto=".1f",    
    aspect="auto",
    color_continuous_scale="Greens",
    title="Dinero Acumulado (Hora vs Mesa)",
    width=700,         # Ancho amplio para que las mesas entren rectas
    height=600          # Alto suficiente para que no se oculte ninguna hora
)
fig_dinero.update_xaxes(
    side='bottom',                 # Pone las mesas en la parte de arriba
    tickangle=270,                # Fuerza a que las etiquetas estén completamente horizontales
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las mesas de la lista
    tickvals=list(range(len(orden_mesas))),
    ticktext=orden_mesas
)

# 6. Configuración del Eje Y (Horas) - COMPLETO Y SIN LÍNEAS
fig_dinero.update_yaxes(
    type='category',
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las horas de la lista
    tickvals=list(range(len(etiquetas_horas))),
    ticktext=etiquetas_horas
)

# 7. Añadir líneas separadoras de jornada
fig_dinero.add_hline(y=3.5, line_dash="dash", line_color="gray", annotation_text=" Inicio PM (Mediodía)", annotation_position="bottom right")
fig_dinero.add_hline(y=15.5, line_dash="dash", line_color="gray", annotation_text=" Madrugada", annotation_position="bottom right")

fig_dinero.show()

In [48]:
pivot_cantidad = df[df['estado'] == 'Facturado'].pivot_table(
    index='hora_entera', 
    columns='mesa', 
    values='cantidad', 
    aggfunc='sum', 
    fill_value=''
)

pivot_cantidad = pivot_cantidad.reindex(index=orden_horas, columns=orden_mesas)

etiquetas_horas = [f"{h}:00" for h in pivot_cantidad.index]


fig_cantidad = px.imshow(
    pivot_cantidad,
    labels=dict(x="Mesa", y="Hora", color="Cantidad"),
    x=pivot_cantidad.columns,
    y=etiquetas_horas,
    text_auto=".1f",    
    aspect="auto",
    color_continuous_scale="ylgn",
    title="Cantidad Acumulada (Hora vs Mesa)",
    width=700,         # Ancho amplio para que las mesas entren rectas
    height=600          # Alto suficiente para que no se oculte ninguna hora
)

fig_cantidad.update_xaxes(
    side='bottom',                 # Pone las mesas en la parte de arriba
    tickangle=270,                # Fuerza a que las etiquetas estén completamente horizontales
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las mesas de la lista
    tickvals=list(range(len(orden_mesas))),
    ticktext=orden_mesas
)

# 6. Configuración del Eje Y (Horas) - COMPLETO Y SIN LÍNEAS
fig_cantidad.update_yaxes(
    type='category',
    showgrid=False,             # Elimina las líneas de guía del fondo
    zeroline=False,             # Elimina la línea base cero
    tickmode='array',           # Fuerza a Plotly a pintar TODAS las horas de la lista
    tickvals=list(range(len(etiquetas_horas))),
    ticktext=etiquetas_horas
)

# 7. Añadir líneas separadoras de jornada
fig_cantidad.add_hline(y=3.5, line_dash="dash", line_color="gray", annotation_text=" Inicio PM (Mediodía)", annotation_position="bottom right")
fig_cantidad.add_hline(y=15.5, line_dash="dash", line_color="gray", annotation_text=" Madrugada", annotation_position="bottom right")

fig_cantidad.show()

In [19]:
conteo_productos = (df[df['estado'] == 'Facturado']
    .groupby('producto')['cantidad']
    .sum()
    .reset_index()
    .sort_values(by='cantidad', ascending=False)
)
conteo_productos.columns = ['Producto', 'Unidades Vendidas']
conteo_productos

,Producto,Unidades Vendidas
0,ALITAS BBQ,3.0
3,CAFE + POLLO,3.0
11,JUGO PAPAYA,3.0
12,LLUVIA DE ALITAS,2.0
4,CAFE PASADO,2.0
5,CESAR DE CASA,1.0
6,CROISSANT DE JAMON Y QUESO,1.0
2,CAFE + CHORIPAN,1.0
1,ARANDANO CON LECHE,1.0
8,FRAPUCCINO CLASICO,1.0
